# 05 – SQLite database and SQL queries from Python

**Owner:** Luca · **Requirement:** data storage in a SQLite database, SQL queries from within Python

This notebook builds the project database `db/sci_alpi.db` from the CSV files of the other notebooks and queries it with SQL.

1. Create the tables with `CREATE TABLE`, including primary keys, foreign keys and `CHECK` rules.
2. Load the CSV files and count the rows before and after loading (input for `docs/integration_log.md`).
3. Check the data with SQL (duplicates, missing values, unmatched keys).
4. Answer project questions with `JOIN`, `GROUP BY`, `HAVING` and a `WITH` query.
5. Check the SQL results against pandas and against Matthias's published numbers.

Run All rebuilds **Luca's tables** from the CSV files. The tables that start with `checkpoint_` (and `pilot_resort_id_crosswalk`) belong to Matthias's snow checkpoints (`python src/run_snow_checkpoints.py`) and are **kept**, so this notebook never deletes the work of a teammate. The snow table for all resorts is **not** included yet, because there is no validated full snow dataset (see `docs/snow_expansion_status.md`).

**Data version.** The `resorts` table now has **293 rows, one per ski area**, from the repaired file `resorts_features.csv` (notebook `02.4_resort_data_repair.ipynb`). The old 300-row table contained seven ski areas that appeared twice and twelve rows typed by hand with national-median numbers; it is no longer used (details in `data/processed/resorts_repair_audit.csv`). Two columns are still *flagged as not verified*: `slope_km` (`slope_status`) and `hotels_4_5_star_10km` (`hotels_status`); see section 3.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Find the repository root (the folder with requirements.txt)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
DB_DIR = ROOT / "db"
DB_DIR.mkdir(exist_ok=True)
DB_PATH = DB_DIR / "sci_alpi.db"

# Rebuild Luca's tables so that every run gives the same result.
# Matthias's tables (checkpoint_*, pilot_resort_id_crosswalk) are NOT touched: the file is never deleted.
LUCA_VIEWS = ["v_latest_earnings"]
LUCA_TABLES = ["snow_winters_pilot", "snow_summary_pilot", "station_winters", "fx_eur_chf_daily",
               "country_stats", "closures", "resorts", "resort_roster"]   # children before parents

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = OFF")
for view in LUCA_VIEWS:
    con.execute(f"DROP VIEW IF EXISTS {view}")
for table in LUCA_TABLES:
    con.execute(f"DROP TABLE IF EXISTS {table}")
con.commit()
con.execute("PRAGMA foreign_keys = ON")   # SQLite does not check foreign keys unless this is switched on
kept = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def sql(query, params=()):
    """Run a SELECT query and return the result as a pandas table."""
    return pd.read_sql_query(query, con, params=params)

print("Repository:", ROOT.name, "| Database:", DB_PATH.relative_to(ROOT))
print("Tables kept from teammates:", kept if kept else "none (run src/run_snow_checkpoints.py later to add them)")

Repository: sci-alpi | Database: db/sci_alpi.db
Tables kept from teammates: ['checkpoint_climate_coefficients', 'checkpoint_climate_model_metrics', 'checkpoint_closure_locations', 'checkpoint_data_quality', 'checkpoint_snow_model_summary', 'checkpoint_snow_model_winters', 'checkpoint_station_tests', 'pilot_resort_id_crosswalk']


## 1. Schema

Each `CREATE TABLE` defines the columns, the **primary key** (a column that is unique for each row), **foreign keys** (a column that must exist in another table) and `CHECK` rules (only allowed values).

Design decisions:
- `resorts` has **one row per ski area** (293). `resort_id` (the last part of the Skiresort URL) is the primary key and a foreign key to `resort_roster`. Ski areas that cross a border (for example Portes du Soleil) appear once, with the country of the main access in `country`, and the list of countries in `country_memberships`.
- Three columns carry a **status**: `price_source` (`scraped` or `manual_entry`), `slope_status` and `hotels_status`. A value that is not verified keeps a flag instead of being silently trusted.
- `ski_routes_km` is separate from `slope_km`: the old scraper had stored the ski-route length in the slope column.
- `closures` has no coordinates yet. Geocoding comes later.
- The pilot snow tables are kept separate, because they contain only 10 resorts and use the saturated indicator.

In [2]:
SCHEMA = '''
CREATE TABLE resort_roster (
    resort_id                       TEXT PRIMARY KEY,
    resort_name                     TEXT NOT NULL,
    url                             TEXT NOT NULL,
    country_memberships             TEXT,
    country_for_analysis            TEXT NOT NULL CHECK (country_for_analysis IN ('AT','CH','FR','IT','cross_border')),
    source_row_count                INTEGER,
    elevation_base_m                INTEGER,
    elevation_top_m                 INTEGER,
    latitude                        REAL,
    longitude                       REAL,
    scope_decision                  TEXT,
    include_in_alpine_candidate_set INTEGER NOT NULL CHECK (include_in_alpine_candidate_set IN (0,1)),
    base_elevation_at_point_verified INTEGER CHECK (base_elevation_at_point_verified IN (0,1)),
    snow_method_validated           INTEGER CHECK (snow_method_validated IN (0,1))
);

CREATE TABLE resorts (
    resort_id               TEXT PRIMARY KEY REFERENCES resort_roster(resort_id),
    resort_name             TEXT NOT NULL,
    country                 TEXT NOT NULL CHECK (country IN ('AT','CH','FR','IT')),
    country_memberships     TEXT,
    cross_border            INTEGER NOT NULL CHECK (cross_border IN (0,1)),
    url                     TEXT NOT NULL,
    elevation_base_m        INTEGER,
    elevation_top_m         INTEGER,
    slope_km                REAL,
    ski_routes_km           REAL,
    slope_status            TEXT NOT NULL CHECK (slope_status IN ('unverified_old_parser','reparsed')),
    lifts_count             REAL,
    rating_stars            REAL,
    price_day_raw           REAL,
    currency                TEXT CHECK (currency IN ('CHF','EUR')),
    price_day_eur           REAL NOT NULL,
    price_source            TEXT NOT NULL CHECK (price_source IN ('scraped','manual_entry')),
    in_alpine_candidate_set INTEGER NOT NULL CHECK (in_alpine_candidate_set IN (0,1)),
    in_alpine_analysis_set  INTEGER NOT NULL CHECK (in_alpine_analysis_set IN (0,1)),
    scope_decision          TEXT,
    alps_review_flag        TEXT,
    latitude                REAL NOT NULL,
    longitude               REAL NOT NULL,
    coordinate_method       TEXT,
    coordinate_check        TEXT,
    geocode_difference_km   REAL,
    nearest_other_resort_km REAL,
    dist_zurich_km          REAL,
    dist_geneva_km          REAL,
    dist_milan_km           REAL,
    dist_turin_km           REAL,
    dist_munich_km          REAL,
    dist_lyon_km            REAL,
    dist_nearest_city_km    REAL,
    hotels_4_5_star_10km    INTEGER,
    hotels_status           TEXT NOT NULL CHECK (hotels_status IN ('not_measured','openstreetmap_overpass')),
    CHECK (elevation_base_m < elevation_top_m)
);

CREATE TABLE closures (
    closure_id    INTEGER PRIMARY KEY,
    facility_name TEXT NOT NULL,
    comune        TEXT,
    province      TEXT,
    region        TEXT NOT NULL,
    status        TEXT NOT NULL CHECK (status IN ('dismesso','chiuso temporaneamente')),
    altitude_text TEXT,
    facility_type TEXT,
    year_built    TEXT,
    year_closed   TEXT,
    source_page   INTEGER
);

CREATE TABLE country_stats (
    country               TEXT NOT NULL CHECK (country IN ('AT','CH','FR','IT')),
    year                  INTEGER NOT NULL,
    pli_eu27              REAL,
    net_earnings_year_eur REAL,
    net_earnings_hour_eur REAL,
    PRIMARY KEY (country, year)
);

CREATE TABLE fx_eur_chf_daily (
    date    TEXT PRIMARY KEY,
    eur_chf REAL NOT NULL            -- CHF per 1 EUR
);

CREATE TABLE station_winters (
    station_id     TEXT NOT NULL,
    station_name   TEXT,
    lat            REAL,
    lon            REAL,
    elev_m         REAL,
    winter         INTEGER NOT NULL,
    period         TEXT NOT NULL CHECK (period IN ('past','recent')),
    snow_days_30cm INTEGER,
    observed_days  INTEGER,
    source         TEXT,
    PRIMARY KEY (station_id, winter)
);

CREATE TABLE snow_summary_pilot (
    resort_id      TEXT PRIMARY KEY,
    name           TEXT,
    country        TEXT,
    elev_base_m    INTEGER,
    days_past      REAL,
    days_recent    REAL,
    lost_snow_days REAL,
    risk_class     TEXT
);

CREATE TABLE snow_winters_pilot (
    resort_id      TEXT NOT NULL,
    winter         INTEGER NOT NULL,
    period         TEXT,
    snow_days_30cm INTEGER,
    elev_query_m   INTEGER,
    PRIMARY KEY (resort_id, winter)
);

-- A view is a saved query: the latest year with net hourly earnings for each country
CREATE VIEW v_latest_earnings AS
SELECT cs.country, cs.year, cs.net_earnings_hour_eur
FROM country_stats cs
JOIN (SELECT country, MAX(year) AS year
      FROM country_stats
      WHERE net_earnings_hour_eur IS NOT NULL
      GROUP BY country) m
  ON m.country = cs.country AND m.year = cs.year;
'''
con.executescript(SCHEMA)
print(sql("SELECT type, name FROM sqlite_master WHERE type IN ('table','view') AND name NOT LIKE 'sqlite_%' ORDER BY type DESC, name").to_string(index=False))

 type                             name
 view                v_latest_earnings
table  checkpoint_climate_coefficients
table checkpoint_climate_model_metrics
table     checkpoint_closure_locations
table          checkpoint_data_quality
table    checkpoint_snow_model_summary
table    checkpoint_snow_model_winters
table         checkpoint_station_tests
table                         closures
table                    country_stats
table                 fx_eur_chf_daily
table        pilot_resort_id_crosswalk
table                    resort_roster
table                          resorts
table               snow_summary_pilot
table               snow_winters_pilot
table                  station_winters


## 2. Load the CSV files

The helper `load()` reads the CSV file, prepares it for the table and appends it with `to_sql`. It returns the number of rows in the file and in the table, so that no loss goes unnoticed.

In [3]:
log = []   # one entry per table: rows in the CSV file vs rows in the database

def load(table, df, source):
    df.to_sql(table, con, if_exists="append", index=False)
    n_db = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    log.append({"table": table, "source": source, "rows_in_file": len(df), "rows_in_table": n_db, "lost": len(df) - n_db})

# 1) Resort roster (one row per ski area)
roster = pd.read_csv(PROCESSED / "resorts_snow_roster.csv")
for col in ["include_in_alpine_candidate_set", "base_elevation_at_point_verified", "snow_method_validated"]:
    roster[col] = roster[col].astype(int)
roster_cols = [c[1] for c in con.execute("PRAGMA table_info(resort_roster)")]
load("resort_roster", roster[roster_cols], "resorts_snow_roster.csv")

# 2) Resorts (293 rows, one per ski area; repaired file from notebook 02.4)
resorts = pd.read_csv(PROCESSED / "resorts_features.csv")
for col in ["cross_border", "in_alpine_candidate_set", "in_alpine_analysis_set"]:
    resorts[col] = resorts[col].astype(int)
resorts_cols = [c[1] for c in con.execute("PRAGMA table_info(resorts)")]
load("resorts", resorts[resorts_cols], "resorts_features.csv")

# 3) Legambiente closures
closures = pd.read_csv(PROCESSED / "legambiente_closures.csv")
closures.insert(0, "closure_id", range(1, len(closures) + 1))
load("closures", closures, "legambiente_closures.csv")

# 4) Country statistics, exchange rate, station winters, pilot snow
load("country_stats", pd.read_csv(PROCESSED / "country_stats.csv"), "country_stats.csv")
load("fx_eur_chf_daily", pd.read_csv(PROCESSED / "fx_eur_chf_daily.csv"), "fx_eur_chf_daily.csv")
stations = pd.read_csv(PROCESSED / "meteoswiss_station_winters.csv")
station_cols = [c[1] for c in con.execute("PRAGMA table_info(station_winters)")]
load("station_winters", stations[station_cols], "meteoswiss_station_winters.csv")
load("snow_summary_pilot", pd.read_csv(PROCESSED / "snow_summary_10resorts.csv"), "snow_summary_10resorts.csv")
load("snow_winters_pilot", pd.read_csv(PROCESSED / "snow_winters_pilot.csv"), "snow_winters_pilot.csv")

con.commit()
load_log = pd.DataFrame(log)
load_log

,table,source,rows_in_file,rows_in_table,lost
0,resort_roster,resorts_snow_roster.csv,293,293,0
1,resorts,resorts_features.csv,293,293,0
2,closures,legambiente_closures.csv,184,184,0
3,country_stats,country_stats.csv,123,123,0
4,fx_eur_chf_daily,fx_eur_chf_daily.csv,4098,4098,0
5,station_winters,meteoswiss_station_winters.csv,400,400,0
6,snow_summary_pilot,snow_summary_10resorts.csv,10,10,0
7,snow_winters_pilot,snow_winters_pilot.csv,20,20,0


## 3. Quality checks with SQL

Four checks. (a) Does every `resorts.resort_id` exist in the roster? (b) Which resorts appear more than once? (c) How many values are missing, and which columns are *not verified*? (d) Are the values plausible (elevation, price, coordinates inside the Alps region)?

In [4]:
# (a) Foreign keys: this must return an empty list
print("Foreign key violations:", con.execute("PRAGMA foreign_key_check").fetchall())

Foreign key violations: []


In [5]:
# (b) Resort IDs that appear in more than one row. GROUP BY + HAVING keeps only the groups with more than one row.
# The primary key already forbids duplicates; the query documents that the old problem (7 repeated areas) is gone.
dups = sql('''
    SELECT resort_id,
           COUNT(*)                 AS n_rows,
           GROUP_CONCAT(country)    AS countries,
           GROUP_CONCAT(price_day_eur) AS prices_eur
    FROM resorts
    GROUP BY resort_id
    HAVING COUNT(*) > 1
    ORDER BY resort_id
''')
print("Repeated resort IDs:", len(dups))
dups

Repeated resort IDs: 0


,resort_id,n_rows,countries,prices_eur


In [6]:
# (c) Missing values and status flags in the resort table
quality = sql('''
    SELECT COUNT(*)                                   AS n_rows,
           SUM(price_day_eur IS NULL)                 AS price_eur_missing,
           SUM(price_source = 'manual_entry')         AS prices_typed_by_hand,
           SUM(price_day_raw IS NULL)                 AS price_raw_missing,
           SUM(lifts_count IS NULL)                   AS lifts_count_missing,
           SUM(slope_status = 'unverified_old_parser') AS slope_km_not_verified,
           SUM(hotels_status = 'not_measured')        AS hotels_not_measured,
           SUM(coordinate_check IN ('two_geocodes_differ_review','single_geocode_only')) AS coordinates_to_review
    FROM resorts
''')
quality

,n_rows,price_eur_missing,prices_typed_by_hand,price_raw_missing,lifts_count_missing,slope_km_not_verified,hotels_not_measured,coordinates_to_review
0,293,0,2,2,1,1,0,0


In [7]:
# (d) Plausibility. Elevation and price are checked for all 293 areas; the position only for the Alpine analysis set,
# because the file also contains Pyrenees, Apennines, Massif Central, Jura and Bohemian Forest areas that are outside the project scope.
bad = sql('''
    SELECT resort_id, scope_decision, alps_review_flag, elevation_base_m, elevation_top_m, price_day_eur, latitude, longitude
    FROM resorts
    WHERE elevation_base_m >= elevation_top_m
       OR price_day_eur NOT BETWEEN 10 AND 200
       OR (in_alpine_analysis_set = 1 AND (latitude  NOT BETWEEN 43 AND 48.5
                                        OR longitude NOT BETWEEN 5 AND 17.5
                                        OR nearest_other_resort_km > 40))
''')
print("Rows that need a look:", len(bad))
bad

Rows that need a look: 0


,resort_id,scope_decision,alps_review_flag,elevation_base_m,elevation_top_m,price_day_eur,latitude,longitude


**Reading the checks.** There are no repeated resorts and no foreign-key problems. The plausibility query returns no rows. It includes an *isolation* rule (an Alpine area more than 40 km from every other area is suspicious), which is how 20 wrong positions of the old table were found (see notebook 02.4). Six areas carry an `alps_review_flag` (Hochficht, Abetone, Cimone, Lelex-Crozet, Saint-Lary-Soulan, Font-Romeu): they lie outside the Alps and are left out of the analysis set `in_alpine_analysis_set`. Two prices are typed by hand (`price_source = 'manual_entry'`, Alta Badia and Galtür), so their raw price is empty. Positions marked `two_geocodes_differ_review` or `single_geocode_only` (14 areas) should be checked by hand, and `geocoders_agree_not_independent` (226 areas) means only that two name-based geocoders agree, which is *not* proof (the isolation rule showed this). The columns `slope_km` and `hotels_4_5_star_10km` are **flagged**, not verified: until `python src/reparse_skiresort.py` and `python src/fetch_hotels_overpass.py` are run in the Codespace, every price-versus-feature analysis that uses them is provisional.

## 4. Project questions in SQL

### 4.1 Price and elevation by country (`GROUP BY`)

In [8]:
q_country = sql('''
    SELECT country,
           COUNT(*)                          AS n_ski_areas,
           ROUND(AVG(price_day_eur), 1)      AS avg_day_pass_eur,
           ROUND(AVG(elevation_base_m))      AS avg_base_m,
           ROUND(AVG(slope_km), 1)           AS avg_slope_km_NOT_VERIFIED,
           ROUND(AVG(rating_stars), 2)       AS avg_rating
    FROM resorts
    GROUP BY country
    ORDER BY avg_day_pass_eur DESC
''')
q_country

,country,n_ski_areas,avg_day_pass_eur,avg_base_m,avg_slope_km_NOT_VERIFIED,avg_rating
0,CH,73,72.7,1285.0,64.8,3.68
1,AT,75,72.4,1058.0,71.2,4.15
2,IT,74,61.0,1317.0,55.3,3.74
3,FR,71,49.3,1335.0,108.8,3.56


**Reading the result.** The averages are over the 293 ski areas of the file (one row each). Switzerland (72.7 EUR) and Austria (72.4 EUR) have the highest average day-pass list prices, Italy follows with 61.0 EUR and France is the cheapest with 49.3 EUR. The column `avg_slope_km_NOT_VERIFIED` is shown only to make the problem visible: the old scraper took the length of the *ski routes* instead of the slopes for many areas, so these averages must not be interpreted until `python src/reparse_skiresort.py` has been run. The group of 293 also contains resorts outside the Alps (Pyrenees, Apennines, Jura), see 4.2.

### 4.2 Ski areas by analysis country, with the roster (`JOIN`)

`JOIN` combines two tables on a common key. Here `resorts` and `resort_roster` are joined on `resort_id`. Every ski area is counted once. The roster has the analysis group of each area, in which cross-border areas have their own category. Only areas in the Alpine candidate set (`include_in_alpine_candidate_set = 1`) are kept.

In [9]:
q_roster = sql('''
    SELECT rr.country_for_analysis                AS country_group,
           COUNT(*)                               AS n_ski_areas,
           SUM(r.cross_border)                    AS n_cross_border_areas,
           ROUND(AVG(r.price_day_eur), 1)         AS avg_day_pass_eur
    FROM resorts r
    JOIN resort_roster rr ON rr.resort_id = r.resort_id
    WHERE rr.include_in_alpine_candidate_set = 1
    GROUP BY rr.country_for_analysis
    ORDER BY n_ski_areas DESC
''')
q_roster

,country_group,n_ski_areas,n_cross_border_areas,avg_day_pass_eur
0,AT,74,0,72.2
1,CH,72,0,72.1
2,IT,69,0,61.4
3,FR,55,0,50.4
4,cross_border,5,5,80.6


**Reading the result.** 275 of the 293 areas are in the Alpine candidate set of the snow roster (AT 74, CH 72, IT 69, FR 55, and 5 cross-border areas). The 18 areas outside the set (15 in the Pyrenees and the Massif Central, 3 in the Apennines) stay in the table with `scope_decision = 'outside_alps'`, so that the group can see what was excluded and why. Six more areas of the candidate set are outside the Alps by geography (`alps_review_flag`); the analysis set `in_alpine_analysis_set` therefore has 269 areas. The 5 cross-border areas (Ischgl-Samnaun, Portes du Soleil, Via Lattea, Espace San Bernardo, Zermatt-Cervinia) have the highest average price (80.6 EUR). With only 5 areas this average is driven by Zermatt-Cervinia (121.8 EUR), so it is a description, not a result.

### 4.3 Affordability: working hours needed for a day pass (SQ4)

`price_day_eur / net_earnings_hour_eur` gives the hours of net work that pay one day pass. The view `v_latest_earnings` picks the latest year with data for each country (2023 for CH, 2025 for AT, FR and IT). The earnings are in nominal euros and the day-pass price is the list price of the scraped site.

In [10]:
q_hours = sql('''
    SELECT r.country,
           e.year                                          AS earnings_year,
           COUNT(*)                                        AS n_resorts,
           ROUND(AVG(r.price_day_eur), 1)                  AS avg_day_pass_eur,
           ROUND(e.net_earnings_hour_eur, 1)               AS net_hourly_earnings_eur,
           ROUND(AVG(r.price_day_eur / e.net_earnings_hour_eur), 2) AS work_hours_per_day_pass
    FROM resorts r
    JOIN v_latest_earnings e ON e.country = r.country
    WHERE r.price_day_eur IS NOT NULL
    GROUP BY r.country
    ORDER BY work_hours_per_day_pass DESC
''')
q_hours

,country,earnings_year,n_resorts,avg_day_pass_eur,net_hourly_earnings_eur,work_hours_per_day_pass
0,IT,2025,74,61.0,15.8,3.86
1,AT,2025,75,72.4,24.1,3.01
2,FR,2025,71,49.3,18.4,2.68
3,CH,2023,73,72.7,46.6,1.56


**Reading the result.** A day pass costs about 3.9 hours of net work in Italy, 3.0 in Austria, 2.7 in France and 1.6 in Switzerland. Switzerland has the highest prices but also much higher net earnings, so it is the most affordable in this measure. Limits: the Swiss earnings are from 2023, the others from 2025; prices are list prices of one day pass; this is a description, not a causal explanation.

### 4.4 Exchange-rate check for the Swiss prices (`WITH`)

`resorts_features.csv` converts CHF to EUR with a fixed factor of 1.05. The Frankfurter series stores CHF per EUR; its average for 2025 gives another factor. The `WITH` query computes this average first and then uses it in the main query.

In [11]:
q_fx = sql('''
    WITH fx AS (
        SELECT AVG(eur_chf) AS avg_chf_per_eur
        FROM fx_eur_chf_daily
        WHERE date >= '2025-01-01'
    )
    SELECT COUNT(*)                                        AS n_chf_prices,
           ROUND(fx.avg_chf_per_eur, 4)                    AS avg_chf_per_eur_2025,
           ROUND(1.0 / fx.avg_chf_per_eur, 4)              AS eur_per_chf_2025,
           ROUND(AVG(r.price_day_eur), 2)                  AS avg_price_eur_fixed_1_05,
           ROUND(AVG(r.price_day_raw / fx.avg_chf_per_eur), 2) AS avg_price_eur_2025_rate,
           ROUND(100.0 * (AVG(r.price_day_raw / fx.avg_chf_per_eur) / AVG(r.price_day_eur) - 1), 1) AS difference_pct
    FROM resorts r, fx
    WHERE r.currency = 'CHF' AND r.price_day_raw IS NOT NULL
''')
q_fx

,n_chf_prices,avg_chf_per_eur_2025,eur_per_chf_2025,avg_price_eur_fixed_1_05,avg_price_eur_2025_rate,difference_pct
0,73,0.937,1.0672,72.74,73.93,1.6


**Reading the result.** With the 2025 average rate, the 73 Swiss prices would be about 1.6 % higher in euros than with the fixed factor 1.05. The effect is small compared with the differences between countries, but the group should use one documented rate. The data in `resorts_features.csv` is not changed here. (73 and not 75 as in the old table: hand-typed Swiss rows were removed in notebook 02.4.)

### 4.5 Closed ski facilities by region (`GROUP BY` with conditional counts)

In [12]:
q_closures = sql('''
    SELECT region,
           SUM(status = 'dismesso')               AS dismissed,
           SUM(status = 'chiuso temporaneamente') AS temporarily_closed,
           COUNT(*)                               AS total
    FROM closures
    GROUP BY region
    ORDER BY total DESC
''')
q_closures

,region,dismissed,temporarily_closed,total
0,Piemonte,38,7,45
1,Lombardia,32,2,34
2,Veneto,16,3,19
3,Emilia-Romagna,12,6,18
4,Abruzzo,10,5,15
5,Friuli Venezia Giulia,7,1,8
6,Toscana,7,0,7
7,Valle d’Aosta,5,1,6
8,Lazio,2,3,5
9,Trentino-Alto Adige,4,0,4


### 4.6 Measured snow days: past vs recent by elevation band

The stations come from `meteoswiss_station_winters.csv` (20 stations, 400 station winters). `CASE WHEN` creates the elevation band. The result is compared with Matthias's published values in section 5.

In [13]:
q_snow = sql('''
    SELECT CASE WHEN elev_m < 1500 THEN 'below 1500 m' ELSE '1500 m and above' END AS elevation_band,
           period,
           COUNT(DISTINCT station_id)         AS n_stations,
           ROUND(AVG(snow_days_30cm), 1)      AS avg_days_30cm
    FROM station_winters
    GROUP BY elevation_band, period
    ORDER BY elevation_band, period DESC
''')
q_snow

,elevation_band,period,n_stations,avg_days_30cm
0,1500 m and above,recent,9,99.5
1,1500 m and above,past,9,106.5
2,below 1500 m,recent,11,33.1
3,below 1500 m,past,11,55.3


**Reading the result.** In the 20 Swiss stations the average number of days with at least 30 cm fell more below 1,500 m (55.3 to 33.1 days) than at 1,500 m and above (106.5 to 99.5 days). These are measured values, not model estimates.

### 4.7 Pilot resorts: which ones can be joined to the roster? (`LEFT JOIN`)

A `LEFT JOIN` keeps all rows of the left table, even without a match. First the pilot table is joined **directly** to the roster by `resort_id`; then it is joined **through the crosswalk table** `pilot_resort_id_crosswalk` that Matthias created (it translates the old pilot names into the canonical Skiresort IDs).

In [14]:
q_pilot = sql('''
    SELECT s.resort_id,
           s.country,
           s.risk_class,
           s.days_past,
           s.days_recent,
           CASE WHEN rr.resort_id IS NULL THEN 'no match' ELSE 'matched' END AS roster_match
    FROM snow_summary_pilot s
    LEFT JOIN resort_roster rr ON rr.resort_id = s.resort_id
    ORDER BY roster_match, s.resort_id
''')
q_pilot

,resort_id,country,risk_class,days_past,days_recent,roster_match
0,aprica,IT,reliable,123.2,113.5,matched
1,les-2-alpes,FR,reliable,128.4,132.1,matched
2,livigno,IT,reliable,135.4,132.9,matched
3,soelden,AT,reliable,136.0,135.8,matched
4,st-moritz-corviglia,CH,reliable,136.1,132.6,matched
5,andermatt-sedrun-disentis,CH,reliable,136.2,136.3,no match
6,breuil-cervinia-valtournenche-zermatt,IT,reliable,131.6,132.2,no match
7,kitzbuehel-kirchberg,AT,reliable,119.0,124.5,no match
8,les-houches-saint-gervais-chamonix,FR,reliable,124.5,135.2,no match
9,zermatt-matterhorn,CH,reliable,136.2,136.3,no match


In [15]:
# Second join: pilot -> crosswalk -> roster (only possible if Matthias's tables are in the database)
has_crosswalk = con.execute("SELECT COUNT(*) FROM sqlite_master WHERE name = 'pilot_resort_id_crosswalk'").fetchone()[0] == 1
if has_crosswalk:
    q_pilot_cw = sql('''
        SELECT s.resort_id                          AS pilot_id,
               cw.canonical_resort_id,
               r.country,
               r.price_day_eur,
               CASE WHEN r.resort_id IS NULL THEN 'no match' ELSE 'matched' END AS match_via_crosswalk
        FROM snow_summary_pilot s
        LEFT JOIN pilot_resort_id_crosswalk cw ON cw.pilot_resort_id = s.resort_id
        LEFT JOIN resorts r                    ON r.resort_id = cw.canonical_resort_id
        ORDER BY match_via_crosswalk, s.resort_id
    ''')
else:
    q_pilot_cw = None
    print("pilot_resort_id_crosswalk not found: run  python src/run_snow_checkpoints.py")
q_pilot_cw

,pilot_id,canonical_resort_id,country,price_day_eur,match_via_crosswalk
0,andermatt-sedrun-disentis,andermatt-oberalp-sedrun,CH,96.6,matched
1,aprica,aprica,IT,57.0,matched
2,breuil-cervinia-valtournenche-zermatt,zermatt-breuil-cervinia-valtournenche-matterhorn,CH,121.8,matched
3,kitzbuehel-kirchberg,kitzski-kitzbuehel-kirchberg,AT,83.0,matched
4,les-2-alpes,les-2-alpes,FR,66.0,matched
5,les-houches-saint-gervais-chamonix,les-houches-saint-gervais-prarion-bellevue-cha...,FR,59.0,matched
6,livigno,livigno,IT,72.0,matched
7,soelden,soelden,AT,86.5,matched
8,st-moritz-corviglia,st-moritz-corviglia,CH,52.5,matched
9,zermatt-matterhorn,zermatt-breuil-cervinia-valtournenche-matterhorn,CH,121.8,matched


**Reading the result.** Directly, only 5 of the 10 pilot resorts match the roster by `resort_id`; the other 5 use an older identifier (for example `zermatt-matterhorn` against `zermatt-breuil-cervinia-valtournenche-matterhorn`). Through the crosswalk all 10 can be joined, and two pilot entries (Zermatt and Cervinia) point to the same connected area, so the 10 pilot entries cover 9 ski areas. All 10 pilot resorts are classed `reliable`, which shows that the pilot indicator does not separate the resorts; it is not used for the risk classes.

## 5. Checks of the SQL results

The SQL results are recomputed with pandas, and the snow result is compared with the values in `docs/matthias_simple_explanation.md`.

In [16]:
# 1) Country summary: SQL vs pandas
check = resorts.groupby("country")["price_day_eur"].mean().round(1)
sql_avg = q_country.set_index("country")["avg_day_pass_eur"]
assert (check.sort_index() == sql_avg.sort_index()).all(), "Country averages differ"

# 2) Number of rows: database vs CSV
assert (load_log["lost"] == 0).all(), "Rows lost while loading"

# 3) Working hours: SQL vs pandas for one country
latest = pd.read_csv(PROCESSED / "country_stats.csv").dropna(subset=["net_earnings_hour_eur"])
latest = latest.sort_values("year").groupby("country").tail(1).set_index("country")["net_earnings_hour_eur"]
tmp = resorts.dropna(subset=["price_day_eur"])
pandas_hours = (tmp["price_day_eur"] / tmp["country"].map(latest)).groupby(tmp["country"]).mean().round(2)
assert (pandas_hours.sort_index() == q_hours.set_index("country")["work_hours_per_day_pass"].sort_index()).all(), "Working hours differ"

# 4) Snow: Matthias reports 55.3 -> 33.1 (11 stations below 1,500 m) and 106.5 -> 99.5 (9 stations)
s = q_snow.set_index(["elevation_band", "period"])
assert s.loc[("below 1500 m", "past"), "avg_days_30cm"] == 55.3 and s.loc[("below 1500 m", "recent"), "avg_days_30cm"] == 33.1
assert s.loc[("1500 m and above", "past"), "avg_days_30cm"] == 106.5 and s.loc[("1500 m and above", "recent"), "avg_days_30cm"] == 99.5
assert s.loc[("below 1500 m", "past"), "n_stations"] == 11 and s.loc[("1500 m and above", "past"), "n_stations"] == 9

# 5) One row per ski area: no repeated IDs, same IDs as the roster, no foreign-key problems
assert len(dups) == 0, "Repeated resort IDs"
assert set(resorts["resort_id"]) == set(roster["resort_id"]) and len(resorts) == len(roster) == 293
assert con.execute("PRAGMA foreign_key_check").fetchall() == []

# 6) The status flags agree with data_status.json (written by notebook 02.4)
import json
status = json.loads((PROCESSED / "data_status.json").read_text())
db_status = {"slope_status": sorted(r[0] for r in con.execute("SELECT DISTINCT slope_status FROM resorts")),
             "hotels_status": sorted(r[0] for r in con.execute("SELECT DISTINCT hotels_status FROM resorts"))}
assert db_status == status, (db_status, status)

print("All checks passed.")

All checks passed.


## 6. Integration summary (copied into `docs/integration_log.md`)

In [17]:
load_log

,table,source,rows_in_file,rows_in_table,lost
0,resort_roster,resorts_snow_roster.csv,293,293,0
1,resorts,resorts_features.csv,293,293,0
2,closures,legambiente_closures.csv,184,184,0
3,country_stats,country_stats.csv,123,123,0
4,fx_eur_chf_daily,fx_eur_chf_daily.csv,4098,4098,0
5,station_winters,meteoswiss_station_winters.csv,400,400,0
6,snow_summary_pilot,snow_summary_10resorts.csv,10,10,0
7,snow_winters_pilot,snow_winters_pilot.csv,20,20,0


In [18]:
con.close()
print("Database saved:", DB_PATH.relative_to(ROOT), f"({DB_PATH.stat().st_size/1024:.0f} KB)")

Database saved: db/sci_alpi.db (932 KB)
